<a href="https://colab.research.google.com/github/zaoa3345678-arch/Aaron-lesson-git/blob/main/SYSH_Week08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 這裡示範直接讀取原始 sysh_lecture_07_processed.csv 並完成清理與對照：
df = pd.read_csv('/content/sample_data/sysh_lecture_07_processed.csv')

In [ ]:
# # 1. 清理缺失值、邏輯矛盾(42000033)與外島(澎湖)，保留台灣本島 254 筆乾淨資料
# df = df_raw.dropna().copy()
# valid_ids = df.groupby('求職者id')['是否為最終選擇'].sum()
# df = df[df['求職者id'].isin(valid_ids[valid_ids == 1].index)]
# df = df[df['求職者的聯絡地址'] != '澎湖'].copy()

In [ ]:
print("=== 【工具 1】敘述性統計：未報到(0) vs. 最終報到(1) ===")
summary = df.groupby('是否為最終選擇')[['總投遞家數', '相隔縣市數', '通勤距離_km']].agg(['count', 'mean', 'std', 'median'])
print(summary)

print("\n--- 交叉分析表：是否同縣市 vs. 最終報到比例 ---")
ct_prop = pd.crosstab(df['是否同縣市'], df['是否為最終選擇'], normalize='index') * 100
print(ct_prop.round(1))

=== 【工具 1】敘述性統計：未報到(0) vs. 最終報到(1) ===
        總投遞家數                            相隔縣市數                             \
        count      mean       std median count      mean       std median   
是否為最終選擇                                                                     
0         164  3.926829  0.943567    4.0   164  3.817073  1.790418    4.0   
1          90  2.822222  1.426694    3.0    90  2.000000  2.016784    1.0   

        通勤距離_km                                 
          count        mean         std median  
是否為最終選擇                                         
0           164  207.195122  118.460466  190.0  
1            90  103.722222  118.607466   60.0  

--- 交叉分析表：是否同縣市 vs. 最終報到比例 ---
是否為最終選擇     0     1
是否同縣市              
0        71.7  28.3
1        12.9  87.1


In [ ]:
print("=== 【工具 2】卡方獨立性檢定 (scipy.stats.chi2_contingency) ===")
ct = pd.crosstab(df['是否同縣市'], df['是否為最終選擇'])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)

print(f"卡方統計量 (Chi2) = {chi2:.4f}")
print(f"p-value = {p_val:.4e}")
print(f"自由度 (dof) = {dof}")
print("期望次數矩陣 (Expected Frequencies):\n", pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(2))

=== 【工具 2】卡方獨立性檢定 (scipy.stats.chi2_contingency) ===
卡方統計量 (Chi2) = 38.6628
p-value = 5.0371e-10
自由度 (dof) = 1
期望次數矩陣 (Expected Frequencies):
 是否為最終選擇       0      1
是否同縣市                 
0        143.98  79.02
1         20.02  10.98


In [ ]:
print("=== 【工具 3】獨立樣本 t 檢定 (scipy.stats.ttest_ind) ===")
dist_0 = df[df['是否為最終選擇'] == 0]['通勤距離_km']
dist_1 = df[df['是否為最終選擇'] == 1]['通勤距離_km']

t_stat, p_val_t = stats.ttest_ind(dist_0, dist_1)
print(f"[通勤距離 km] t 統計量 = {t_stat:.4f}, p-value = {p_val_t:.4e}")

# 同時比較使用「相隔縣市數 (1, 2, 3...)」的結果：
step_0 = df[df['是否為最終選擇'] == 0]['相隔縣市數']
step_1 = df[df['是否為最終選擇'] == 1]['相隔縣市數']
t_step, p_step = stats.ttest_ind(step_0, step_1)
print(f"[相隔縣市數] t 統計量 = {t_step:.4f}, p-value = {p_step:.4e}")

=== 【工具 3】獨立樣本 t 檢定 (scipy.stats.ttest_ind) ===
[通勤距離 km] t 統計量 = 6.6556, p-value = 1.7499e-10
[相隔縣市數] t 統計量 = 7.3934, p-value = 2.1128e-12


In [ ]:
print("=== 【工具 4】單因子變異數分析 ANOVA (scipy.stats.f_oneway) ===")
g_north = df[df['求職者地區'] == '北部']['通勤距離_km']
g_central = df[df['求職者地區'] == '中部']['通勤距離_km']
g_south = df[df['求職者地區'] == '南部']['通勤距離_km']
g_east = df[df['求職者地區'] == '東部']['通勤距離_km']

f_stat, p_val_f = stats.f_oneway(g_north, g_central, g_south, g_east)
print(f"[四地區 vs 通勤距離] F 統計量 = {f_stat:.4f}, p-value = {p_val_f:.4e}")
print(df.groupby('求職者地區')['通勤距離_km'].agg(['count', 'mean', 'std']).round(2))

# 也可檢驗四地區的「總投遞家數」差異：
f_app, p_app = stats.f_oneway(
    df[df['求職者地區'] == '北部']['總投遞家數'],
    df[df['求職者地區'] == '中部']['總投遞家數'],
    df[df['求職者地區'] == '南部']['總投遞家數'],
    df[df['求職者地區'] == '東部']['總投遞家數']
)
print(f"\n[四地區 vs 總投遞家數] F 統計量 = {f_app:.4f}, p-value = {p_app:.4f}")

=== 【工具 4】單因子變異數分析 ANOVA (scipy.stats.f_oneway) ===
[四地區 vs 通勤距離] F 統計量 = 20.4497, p-value = 6.9648e-12
       count    mean     std
求職者地區                       
中部        70  131.93   81.98
北部        66  115.30  121.81
南部        75  195.33  135.75
東部        43  274.88  114.40

[四地區 vs 總投遞家數] F 統計量 = 2.6938, p-value = 0.0466


In [ ]:
print("=== 【工具 5】皮爾森相關係數 (scipy.stats.pearsonr) ===")
# 1. 驗證兩種距離指標的一致性：相隔縣市數 vs 通勤距離_km
r_dist, p_dist = stats.pearsonr(df['相隔縣市數'], df['通勤距離_km'])
print(f"1. 相隔縣市數 vs 通勤距離(km): r = {r_dist:.4f}, p = {p_dist:.4e}")

# 2. 通勤距離_km vs 是否為最終選擇 (0/1)
r_choice, p_choice = stats.pearsonr(df['通勤距離_km'], df['是否為最終選擇'])
print(f"2. 通勤距離(km) vs 是否為最終選擇: r = {r_choice:.4f}, p = {p_choice:.4e}")

# 3. 課堂思維碰撞題：總投遞家數 vs 是否為最終選擇 (0/1)
r_apps, p_apps = stats.pearsonr(df['總投遞家數'], df['是否為最終選擇'])
print(f"3. 總投遞家數 vs 是否為最終選擇: r = {r_apps:.4f}, p = {p_apps:.4e}")

=== 【工具 5】皮爾森相關係數 (scipy.stats.pearsonr) ===
1. 相隔縣市數 vs 通勤距離(km): r = 0.7510, p = 2.5321e-47
2. 通勤距離(km) vs 是否為最終選擇: r = -0.3867, p = 1.7499e-10
3. 總投遞家數 vs 是否為最終選擇: r = -0.4225, p = 2.0270e-12


In [ ]:
print("=== 【工具 6】傳統簡單線性迴歸 (scipy.stats.linregress) ===")
res = stats.linregress(df['通勤距離_km'], df['是否為最終選擇'])

print(f"截距 (Intercept, b0) = {res.intercept:.4f}")
print(f"斜率 (Slope, b1)     = {res.slope:.6f}")
print(f"判定係數 (R-squared) = {res.rvalue**2:.4f}")
print(f"p-value              = {res.pvalue:.4e}")
print(f"\n迴歸方程式：預測報到機率 Y = {res.intercept:.4f} + ({res.slope:.6f}) * 通勤距離(km)")

# 課堂震撼教育：請學生代數字進去預測！
for test_km in [0, 100, 300, 430]:
    pred_y = res.intercept + res.slope * test_km
    print(f"當通勤距離 = {test_km:3d} km 時 -> 模型預測報到機率 Y = {pred_y:.3f} ({pred_y*100:.1f}%)")

=== 【工具 6】傳統簡單線性迴歸 (scipy.stats.linregress) ===
截距 (Intercept, b0) = 0.6007
斜率 (Slope, b1)     = -0.001445
判定係數 (R-squared) = 0.1495
p-value              = 1.7499e-10

迴歸方程式：預測報到機率 Y = 0.6007 + (-0.001445) * 通勤距離(km)
當通勤距離 =   0 km 時 -> 模型預測報到機率 Y = 0.601 (60.1%)
當通勤距離 = 100 km 時 -> 模型預測報到機率 Y = 0.456 (45.6%)
當通勤距離 = 300 km 時 -> 模型預測報到機率 Y = 0.167 (16.7%)
當通勤距離 = 430 km 時 -> 模型預測報到機率 Y = -0.021 (-2.1%)
